In [1]:
!pip install biopython pdbfixer

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import json
import requests
import time
import random
import csv

import Bio
import Bio.PDB
import Bio.SeqRecord
from Bio.PDB import PDBParser, PDBIO
from pdbfixer import PDBFixer
from openmm.app import PDBFile

amino_acids = ["A","R","N","D","C","E","Q","G","H","I","L","K","M","F","P","S","T","W","Y","V"]
aa_codes = {
        "A": "ALA",
        "R": "ARG",
        "N": "ASN",
        "D": "ASP",
        "C": "CYS",
        "Q": "GLN",
        "E": "GLU",
        "G": "GLY",
        "H": "HIS",
        "I": "ILE",
        "L": "LEU",
        "K": "LYS",
        "M": "MET",
        "F": "PHE",
        "P": "PRO",
        "S": "SER",
        "T": "THR",
        "W": "TRP",
        "Y": "TYR",
        "V": "VAL"
    }

qmean_url = "https://swissmodel.expasy.org/qmean/submit/"
path_1LYZ = "./1LYZ.pdb"

### Parse PDB File

In [3]:
def getProteinSequenceFromFile(pdbPath: str,pdbcode = "1LYZ") -> str:
    
    

    pdbparser = Bio.PDB.PDBParser(QUIET=True)   # suppress PDBConstructionWarning
    struct = pdbparser.get_structure(pdbcode, pdbPath)

    
    ppb = Bio.PDB.PPBuilder()
    seqrecords = []
    for i, chain in enumerate(struct.get_chains()):
        # extract and store sequences as list of SeqRecord objects
        pps = ppb.build_peptides(chain)    # polypeptides
        seq = Bio.Seq.Seq("".join([str(i.get_sequence()) for i in pps]))
        # seqid = pdbcode + chain.id
        # seqrec = Bio.SeqRecord.SeqRecord(seq, id=seqid, description="Sequence #{}, {}".format(i+1, seqid))
        # seqrecords.append(seqrec)
    return str(seq)
getProteinSequenceFromFile(path_1LYZ)

'KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL'

## Calculate Q-Mean using the qmean API

In [4]:
def getQMean(pdbPath: str) -> str:
    poll_interval = 10
    timeout = 600
    #Queries the qmean api with our file
    response = requests.post(url=qmean_url,
                             data={ 
                                "email": "andrewsvester@gmail.com" 
                             },
                             files={
                                "structure": open(pdbPath, 'rb')
                             })

    #Checks to see if the http request was successful
    response.raise_for_status()

    results_url = response.json()["results_json"]

    print(f"Submitted {pdbPath}")
    print(f"Checking: {results_url}")

    start_time = time.time()
    
    #loop to wait untill we get the response correctly
    while True:
        try:
            status_response = requests.get(
                results_url,
                timeout=30
            )
            status_response.raise_for_status()

            results = status_response.json()

        except requests.exceptions.RequestException as e:
            print("Network error:", e)
            print("Retrying in", poll_interval, "seconds...")
            time.sleep(poll_interval)
            continue

        status = results.get("status")

        print(f"Status: {status}")
        
        if status == "COMPLETED":
            qmean = results["models"]["model_001"]["scores"]["global_scores"]

            qmean_disco = qmean["avg_local_score"]

            print(f"QMEANDisCo: {qmean_disco:.4f}")

            return qmean_disco
        
        if status in ["FAILED", "ERROR"]:
            raise RuntimeError(f"QMEAN processing failed: {results}")
            
        if time.time() - start_time > 300:
            raise TimeoutError(
                f"QMEAN did not finish within {timeout} seconds"
            )
        
        time.sleep(1)

In [40]:
print(getQMean(path_1LYZ))

Submitted ./1LYZ.pdb
Checking: https://swissmodel.expasy.org/qmean/pp9adK.json
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: COMPLETED
QMEANDisCo: 0.7850
0.7849775509623591


### Mutate Sequence by 1 Amino Acid

In [5]:
def mutate(seq):
    i = random.randrange(len(seq))
    a = random.randrange(20)
    while seq[i] == amino_acids[a]:
        a = random.randrange(20)
    
    old_aa = seq[i]
    new_aa = amino_acids[a]
    seq = list(seq)
    seq[i] = amino_acids[a]
    seq = "".join(seq)
    return seq, i, new_aa, old_aa

mutate("KVFGRCELAAAMKRHGLDNY")

('KVFGRCEKAAAMKRHGLDNY', 7, 'K', 'L')

### Modify PDB File Keeping the 3d structure but changing the sequence

In [6]:


"""
Mutate one residue in a PDB structure.

Parameters
----------
input_pdb : str
    Original PDB file.

output_pdb : str
    Where to save the mutated PDB.

position : int
    Zero-based sequence position.

new_aa : str
    One-letter amino acid code.

chain_id : str
    Chain containing the sequence.
"""
def mutate_pdb(input_pdb, output_pdb, position, new_aa, chain_id="A"):
    

    new_resname = aa_codes[new_aa.upper()]

    # Read structure
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", input_pdb)

    model = structure[0]
    chain = model[chain_id]

    # Get standard amino-acid residues
    residues = [
        residue
        for residue in chain
        if residue.id[0] == " "
    ]

    if position < 0 or position >= len(residues):
        raise IndexError(
            f"Position {position} is outside chain {chain_id} "
            f"(length {len(residues)})"
        )

    residue = residues[position]

    old_resname = residue.resname

    print(
        f"Mutating chain {chain_id}, "
        f"position {position + 1}: "
        f"{old_resname} -> {new_resname}"
    )

    # Change residue identity
    residue.resname = new_resname

    # Save temporary PDB
    temp_pdb = output_pdb + ".temp.pdb"

    io = PDBIO()
    io.set_structure(structure)
    io.save(temp_pdb)

    # Use PDBFixer to add atoms missing from the new residue
    fixer = PDBFixer(filename=temp_pdb)

    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()

    fixer.findMissingAtoms()
    fixer.addMissingAtoms()

    with open(output_pdb, "w") as output_file:
        PDBFile.writeFile(
            fixer.topology,
            fixer.positions,
            output_file
        )

    return output_pdb



### Random Walk

In [38]:
seq = getProteinSequenceFromFile(path_1LYZ)
mutations = 0
max_steps = 2
mutation_folder = "./1LYZ_Mutations/"
cur_seq_path = path_1LYZ
cur_pdb_mut_path = mutation_folder + str(mutations) + ".pdb"
cur_high_qmean = getQMean(cur_seq_path)
loops = 0
results = []
while mutations < max_steps and loops <= 100:
    new_seq , position, new_aa, old_aa = mutate(seq)
    mutate_pdb(cur_seq_path, cur_pdb_mut_path, position, new_aa, chain_id="A")
    cur_qmean = getQMean(cur_pdb_mut_path)
    if cur_qmean > (cur_high_qmean - .01):
        mutations += 1
        cur_seq_path = cur_pdb_mut_path
        cur_pdb_mut_path = mutation_folder + str(mutations) + ".pdb"
        seq = new_seq
        results.append([mutation_folder + str(mutations) + ".pdb", cur_qmean, loops])
    if cur_qmean > cur_high_qmean:
        cur_high_qmean = cur_qmean
    loops += 1
    
print(results)

Submitted ./1LYZ.pdb
Checking: https://swissmodel.expasy.org/qmean/PR45N2.json
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: COMPLETED
QMEANDisCo: 0.7850
Mutating chain A, position 58: ILE -> PHE
Submitted ./1LYZ_Mutations/0.pdb
Checking: https://swissmodel.expasy.org/qmean/GzAzMn.json
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
S

### Final Pipeline

In [26]:
getQMean("./1LYZ_Mutations/1.pdb")

Submitted ./1LYZ_Mutations/1.pdb
Checking: https://swissmodel.expasy.org/qmean/wqDVk7.json
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: COMPLETED
QMEANDisCo: 0.7779


0.7778911288236385

In [47]:
with open('results.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile, delimiter=',',
                            quotechar='|', quoting=csv.QUOTE_MINIMAL)
    for row in results:
        writer.writerow(row)


### Reduced Alphabet
Couldn't find the mapping used in the original paper. Am attempting to use the one in this paper instead https://arxiv.org/pdf/cond-mat/0010244

(L FI) (MVW CY) (HA ) (TGPRQS NED) (K)

In [11]:
L = ['L', 'F', 'I']
K = ['K']
A = ['H', 'A']
W = ['M','V','W','C','Y']
S = ['T','G','P','R','Q','S','N','E','D']



def reduce(seq: str) -> str:
    seq = list(seq)
    newseq = ''
    for i in seq:
        if i in L:
            newseq += 'L'
        if i in A:
            newseq += 'A'
        if i in W:
            newseq +='W'
        if i in S:
            newseq += 'S'
        if i in K:
            newseq += 'K'
        
    return newseq
reduce('KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAK')

'KWLSSWSLAAAWKSASLSSWSSWSLSSWWWAAK'

In [10]:
seq = getProteinSequenceFromFile(path_1LYZ)
seq = reduce(seq)
mutate_pdb(path_1LYZ, './reduced_1LYZ.pdb', 0, seq[0], chain_id="A")
for i in range(1, len(seq)):
    mutate_pdb('./reduced_1LYZ.pdb', './reduced_1LYZ.pdb', i, seq[i], chain_id="A")

Mutating chain A, position 1: LYS -> LYS
Mutating chain A, position 2: VAL -> MET
Mutating chain A, position 3: PHE -> LEU
Mutating chain A, position 4: GLY -> ASP
Mutating chain A, position 5: ARG -> ASP
Mutating chain A, position 6: CYS -> MET
Mutating chain A, position 7: GLU -> ASP
Mutating chain A, position 8: LEU -> LEU
Mutating chain A, position 9: ALA -> ALA
Mutating chain A, position 10: ALA -> ALA
Mutating chain A, position 11: ALA -> ALA
Mutating chain A, position 12: MET -> MET
Mutating chain A, position 13: LYS -> LYS
Mutating chain A, position 14: ARG -> ASP
Mutating chain A, position 15: HIS -> ALA
Mutating chain A, position 16: GLY -> ASP
Mutating chain A, position 17: LEU -> LEU
Mutating chain A, position 18: ASP -> ASP
Mutating chain A, position 19: ASN -> ASP
Mutating chain A, position 20: TYR -> MET
Mutating chain A, position 21: ARG -> ASP
Mutating chain A, position 22: GLY -> ASP
Mutating chain A, position 23: TYR -> MET
Mutating chain A, position 24: SER -> ASP
M

KeyboardInterrupt: 


KeyboardInterrupt



In [ ]:
getQMean('./reduced_1LYZ.pdb')

In [12]:
seq = getProteinSequenceFromFile("./1UBQ.pdb")
seq = reduce(seq)
mutate_pdb(path_1LYZ, './reduced_1UBQ.pdb', 0, seq[0], chain_id="A")
for i in range(1, len(seq)):
    mutate_pdb('./reduced_1UBQ.pdb', './reduced_1UBQ.pdb', i, seq[i], chain_id="A")

Mutating chain A, position 1: LYS -> TRP
Mutating chain A, position 2: VAL -> SER
Mutating chain A, position 3: PHE -> LEU
Mutating chain A, position 4: GLY -> LEU
Mutating chain A, position 5: ARG -> TRP
Mutating chain A, position 6: CYS -> LYS
Mutating chain A, position 7: GLU -> SER
Mutating chain A, position 8: LEU -> LEU
Mutating chain A, position 9: ALA -> SER
Mutating chain A, position 10: ALA -> SER
Mutating chain A, position 11: ALA -> LYS
Mutating chain A, position 12: MET -> SER
Mutating chain A, position 13: LYS -> LEU
Mutating chain A, position 14: ARG -> SER
Mutating chain A, position 15: HIS -> LEU
Mutating chain A, position 16: GLY -> SER
Mutating chain A, position 17: LEU -> TRP
Mutating chain A, position 18: ASP -> SER
Mutating chain A, position 19: ASN -> SER
Mutating chain A, position 20: TYR -> SER
Mutating chain A, position 21: ARG -> SER
Mutating chain A, position 22: GLY -> SER
Mutating chain A, position 23: TYR -> LEU
Mutating chain A, position 24: SER -> SER
M

In [13]:
getQMean('./reduced_1UBQ.pdb')

Submitted ./reduced_1UBQ.pdb
Checking: https://swissmodel.expasy.org/qmean/zUMLXn.json
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: QUEUEING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: RUNNING
Status: CO

0.3911537281234864

In [14]:
seq = getProteinSequenceFromFile("./2TRX.pdb")
seq = reduce(seq)
mutate_pdb(path_1LYZ, './reduced_2TRX.pdb', 0, seq[0], chain_id="A")
for i in range(1, len(seq)):
    mutate_pdb('./reduced_2TRX.pdb', './reduced_2TRX.pdb', i, seq[i], chain_id="A")
getQMean('./reduced_2TRX.pdb')

Mutating chain A, position 1: LYS -> SER
Mutating chain A, position 2: VAL -> SER
Mutating chain A, position 3: PHE -> LYS
Mutating chain A, position 4: GLY -> LEU
Mutating chain A, position 5: ARG -> LEU
Mutating chain A, position 6: CYS -> ALA
Mutating chain A, position 7: GLU -> LEU
Mutating chain A, position 8: LEU -> SER
Mutating chain A, position 9: ALA -> SER
Mutating chain A, position 10: ALA -> SER
Mutating chain A, position 11: ALA -> SER
Mutating chain A, position 12: MET -> LEU
Mutating chain A, position 13: LYS -> SER
Mutating chain A, position 14: ARG -> SER
Mutating chain A, position 15: HIS -> SER
Mutating chain A, position 16: GLY -> TRP
Mutating chain A, position 17: LEU -> LEU
Mutating chain A, position 18: ASP -> LYS
Mutating chain A, position 19: ASN -> ALA
Mutating chain A, position 20: TYR -> SER
Mutating chain A, position 21: ARG -> SER
Mutating chain A, position 22: GLY -> ALA
Mutating chain A, position 23: TYR -> LEU
Mutating chain A, position 24: SER -> LEU
M

0.2608234124711541